# Imports

In [1]:
import torch
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import torch.nn as nn
from torch import optim
import torchvision.transforms.functional as F

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

Using Device: cpu


# Start

## Load

In [3]:
data_path = "./data/EMNIST_data"

if os.path.exists(data_path) and os.path.isdir(data_path):
    download = False
    print("EMNIST Data folder found locally. Loading from local.\n")
else:
    download = True
    print("EMNIST Data folder not found locally. Downloading data.\n")

EMNIST Data folder not found locally. Downloading data.



In [4]:
train_dataset = datasets.EMNIST(
    root=data_path,
    split="letters",
    download=download,
    train=True
)

test_dataset = datasets.EMNIST(
    root=data_path,
    split="letters",
    train=False,
    download=download
)

100.0%


## Transform

In [5]:
# Precomputed mean and std for EMNIST Letters dataset
mean = (0.1736,)
std = (0.3317,)

# Create a transform that converts images to tensors and normalizes them
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts images to PyTorch tensors and scales pixel values to [0, 1]
    transforms.Normalize(mean=mean, std=std)   # Applies normalization using the computed mean and std
])

In [7]:
# transformations are applied automatically each time a sample is loaded from the dataset
train_dataset.transform = transform
test_dataset.transform = transform

## Data Loaders - to standardize batching

In [16]:
def create_emnist_dataloaders(train_dataset, test_dataset, batch_size=64):
    """
    Creates DataLoader objects for the EMNIST training and testing datasets.
    Returns:
        tuple (torch.utils.data.DataLoader, torch.utils.data.DataLoader):
            A tuple containing the training and testing DataLoaders (train_loader, test_loader).
    """

    train_dataloader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True
    )  

    test_dataloader = DataLoader(
        dataset=test_dataset,
        batch_size=batch_size,
        shuffle=False
    )  

    return train_dataloader, test_dataloader

In [43]:
train_loader, test_loader = create_emnist_dataloaders(
    train_dataset, test_dataset, batch_size=64
)

## Model

In [9]:
# GRADED FUNCTION: initialize_emnist_model


def initialize_emnist_model(num_classes=26):
    """
    Initializes a sequential neural network model for EMNIST classification.

    Returns:
        tuple: A tuple containing the model, loss function, and optimizer.
               (model, loss_function, optimizer)
    """

    torch.manual_seed(42)

    # Define the neural network model using nn.Sequential
    model = nn.Sequential(
        # Flatten the input images
        nn.Flatten(),
        # 1
        nn.Linear(784, 256),
        nn.ReLU(),
        # 2
        nn.Linear(256, 128),
        nn.ReLU(),

        nn.Linear(128, 26),
    )

    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    return model, loss_function, optimizer

In [10]:
model, loss_func, optimizer = initialize_emnist_model(num_classes=26)

In [11]:
model

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=26, bias=True)
)

In [12]:
loss_func

CrossEntropyLoss()

In [13]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

## Training - One batch

### one batch

In [17]:
train_dataloader, test_dataloader = create_emnist_dataloaders(train_dataset, test_dataset)

In [18]:
batch_idx, (features, targets) = next(enumerate(train_dataloader))

In [20]:
inputs, targets = features.to(DEVICE), targets.to(DEVICE)

In [ ]:
targets = targets - 1 # mnist quirk

In [26]:
optimizer.zero_grad()
outputs = model(inputs)
loss = loss_func(outputs, targets)
loss.backward()
optimizer.step()

In [28]:
loss_value = loss.item()
loss_value

3.26397705078125

In [34]:
outputs.shape

torch.Size([64, 26])

In [37]:
len(torch.argmax(outputs, dim = 1))

64

## Training - Full

In [41]:
def train_epoch(model, loss_function, optimizer, train_loader, device, verbose=True):
    """
    Trains the model for one epoch and calculates the average loss and accuracy.

    Args:
        model (nn.Module): The PyTorch model to be trained.
        loss_function (nn.Module): The loss function used for training.
        optimizer (optim.Optimizer): The optimizer used for updating model parameters.
        train_loader (DataLoader): DataLoader for the training dataset, providing batches of data.
        device (torch.device): The device (CPU or CUDA) where the model and data will be moved.
        verbose (bool, optional): If True, prints the average loss and accuracy for the epoch. Defaults to True.

    Returns:
        tuple: A tuple containing:
            - nn.Module: The trained model after one epoch.
            - float: The average loss for the epoch.
            - float: The accuracy percentage for the epoch.

    Note:
        - The target labels in the `train_loader` (EMNIST letters) are 1-indexed and so the function adjusts them
          to 0-indexed before calculating the loss.
    """

    model.to(device)
    model.train()
    running_loss = 0.0
    
    num_correct_predictions = 0
    total_predictions = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
  
        inputs, targets = inputs.to(device), targets.to(device)
        targets = targets - 1
        optimizer.zero_grad()

        outputs = model(inputs)

        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()
  
        loss_value = loss.item()
        running_loss = running_loss + loss_value
        
        predicted_indices = torch.argmax(outputs, dim=1)
        
        correct_predictions = predicted_indices == targets
        num_correct_in_batch = sum(correct_predictions)
        num_correct_predictions = num_correct_predictions + num_correct_in_batch
        batch_size = len(targets)
        total_predictions = total_predictions + batch_size
    
    average_loss = running_loss / batch_idx
    accuracy_percentage = num_correct_predictions * 100.0 / total_predictions

    if verbose:
        print(
            f"Epoch Loss (Avg): {average_loss:.3f} | Epoch Acc: {accuracy_percentage:.2f}%"
        )
    
    return model, average_loss

In [44]:
model, loss_function, optimizer = initialize_emnist_model(num_classes=26)

model_one_train_epoch, _ = train_epoch(
    model=model,  # model
    loss_function=loss_function,  # loss_function
    optimizer=optimizer,  # optimizer
    train_loader=train_loader,  # train_loader
    device=DEVICE,
)

Epoch Loss (Avg): 0.601 | Epoch Acc: 81.50%


## Eval

In [45]:
def evaluate(model, test_loader, device, verbose=True):
    """
    Evaluates the model on the test dataset and returns the accuracy percentage.

    Args:
        model (nn.Module): The PyTorch model to be evaluated.
        test_loader (DataLoader): DataLoader for the testing dataset.
        device (torch.device): The device to use (CPU or CUDA).
        verbose (bool, optional): If True, prints the test accuracy. Defaults to True.

    Returns:
        float: The accuracy percentage of the model on the test dataset.

    Note:
        - The target labels in the `test_loader` (EMNIST letters) are 1-indexed and so the function adjusts them
          to 0-indexed before calculating the accuracy.
    """

    model.eval()
    num_correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():

        for inputs, targets in test_loader:

            inputs, targets = inputs.to(device), targets.to(device)
            targets = targets - 1
            
            outputs = model(inputs)
            predicted_indices = torch.argmax(outputs, dim=1)

            correct_predictions = predicted_indices == targets
            num_correct_in_batch = sum(correct_predictions)
            num_correct_predictions = num_correct_predictions + num_correct_in_batch
            batch_size = len(targets)
            total_predictions = total_predictions + batch_size

        accuracy_percentage = (num_correct_predictions / total_predictions) * 100

    if verbose:
        print((f"Test Accuracy: {accuracy_percentage:.2f}%"))

    return accuracy_percentage

In [46]:
model_evaluate = evaluate(model, test_loader, DEVICE)

Test Accuracy: 87.26%
